In [ ]:
# ============================================
# PROFESSIONAL ML SOLUTION
# Root Cause: Wrong column names + No encoding
# Solution: Proper data preparation + validation
# ============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, recall_score, f1_score,
                             accuracy_score, precision_score, confusion_matrix,
                             classification_report, roc_curve)
import matplotlib.pyplot as plt

print("="*80)
print("PROFESSIONAL ML SOLUTION: AUC > 85% FOR MATERNITY READMISSION")
print("="*80)

# ============================================
# PHASE 0: DATA EXPLORATION & VALIDATION
# ============================================
print("\n" + "="*80)
print("PHASE 0: DATA EXPLORATION & VALIDATION (CRITICAL)")
print("="*80)

# Load raw data
df = pd.read_csv('/content/test_super.csv')

print(f"\n✓ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Column names inspection
print("\n✓ Column names in your CSV:")
print(df.columns.tolist())

# Data types
print("\n✓ Data types:")
print(df.dtypes)

# Check target variable
print("\n✓ Target variable (checking for 'Readmitted' or similar):")
target_col = None
for col in df.columns:
    if 'readmit' in col.lower():
        target_col = col
        print(f"  Found: {col}")
        print(f"  Values: {df[col].unique()}")
        print(f"  Distribution: {df[col].value_counts()}")
        break

if target_col is None:
    print("  ERROR: No readmission column found!")
    print("  Available columns:", df.columns.tolist())

# Missing values
print("\n✓ Missing values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  No missing values")

# Numeric vs categorical
print("\n✓ Numeric columns:")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_cols)

print("\n✓ Categorical columns:")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(categorical_cols)
for col in categorical_cols:
    if col.lower() != target_col.lower():
        print(f"  {col}: {df[col].unique()[:5]}")

# ============================================
# PHASE 1: DATA PREPARATION
# ============================================
print("\n" + "="*80)
print("PHASE 1: DATA PREPARATION (ENCODING + CLEANING)")
print("="*80)

df_processed = df.copy()

# Step 1: Identify and encode categorical features
print("\n✓ Step 1: Encoding categorical variables")

categorical_features = df_processed.select_dtypes(include=['object']).columns.tolist()

encoders = {}
for col in categorical_features:
    if col.lower() == target_col.lower():
        continue  # Don't encode target yet

    print(f"  Encoding {col}...")
    le = LabelEncoder()
    df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col])
    encoders[col] = le
    print(f"    {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Step 2: Encode target variable (Readmitted: Yes/No → 1/0)
print(f"\n✓ Step 2: Encoding target variable")

if target_col:
    if df_processed[target_col].dtype == 'object':
        # Text target (Yes/No)
        df_processed['Readmitted_binary'] = (df_processed[target_col] == 'Yes').astype(int)
    else:
        # Already numeric
        df_processed['Readmitted_binary'] = df_processed[target_col].astype(int)

    print(f"  {target_col} → Readmitted_binary (0/1)")
    print(f"  Distribution: {df_processed['Readmitted_binary'].value_counts().to_dict()}")
    print(f"  Readmission rate: {df_processed['Readmitted_binary'].mean():.1%}")

# Step 3: Handle missing values
print(f"\n✓ Step 3: Handling missing values")

numeric_features_all = df_processed.select_dtypes(include=[np.number]).columns.tolist()
numeric_features_all.remove('Readmitted_binary')

if df_processed[numeric_features_all].isnull().sum().sum() > 0:
    print(f"  Found missing values, filling with median...")
    df_processed[numeric_features_all] = df_processed[numeric_features_all].fillna(
        df_processed[numeric_features_all].median()
    )
    print(f"  ✓ Filled")
else:
    print(f"  No missing values")

# Step 4: Select final features
print(f"\n✓ Step 4: Selecting final features")

# Use all numeric features
X = df_processed.select_dtypes(include=[np.number]).drop(
    ['Readmitted_binary', target_col], axis=1, errors='ignore'
)
#X = df_processed.select_dtypes(include=[np.number]).drop('Readmitted_binary', axis=1)
y = df_processed['Readmitted_binary']

print(f"  Features selected: {X.shape[1]}")
print(f"  Feature list: {X.columns.tolist()}")
print(f"  Target shape: {y.shape}")

# ============================================
# PHASE 2: EXPLORATORY DATA ANALYSIS
# ============================================
print("\n" + "="*80)
print("PHASE 2: DATA QUALITY CHECKS")
print("="*80)

# Feature statistics
print(f"\n✓ Feature statistics:")
print(X.describe())

# Correlation with target
print(f"\n✓ Feature correlation with target:")
correlations = X.corrwith(y).sort_values(ascending=False)
print(correlations)

# Identify strong vs weak features
strong_features = correlations[correlations > 0.1].index.tolist()
weak_features = correlations[correlations <= 0.1].index.tolist()

print(f"\n  Strong features ({len(strong_features)}): {strong_features}")
print(f"  Weak features ({len(weak_features)}): {weak_features}")

# ============================================
# PHASE 3: FEATURE ENGINEERING
# ============================================
print("\n" + "="*80)
print("PHASE 3: FEATURE ENGINEERING")
print("="*80)

X_engineered = X.copy()

# Add polynomial features for strong features
print(f"\n✓ Adding polynomial features...")

for col in strong_features:
    if len(strong_features) > 0:
        X_engineered[f'{col}_squared'] = X[col] ** 2
        X_engineered[f'{col}_sqrt'] = np.sqrt(np.abs(X[col]) + 1)

# Add interaction features for top features
print(f"✓ Adding interaction features...")

if len(strong_features) >= 2:
    for i, col1 in enumerate(strong_features[:3]):  # Top 3 features
        for col2 in strong_features[i+1:3]:
            X_engineered[f'{col1}_x_{col2}'] = X[col1] * X[col2]

print(f"  Features: {X.shape[1]} → {X_engineered.shape[1]}")

# ============================================
# PHASE 4: DATA STANDARDIZATION
# ============================================
print("\n" + "="*80)
print("PHASE 4: STANDARDIZATION")
print("="*80)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_engineered)
X_scaled = pd.DataFrame(X_scaled, columns=X_engineered.columns)

print(f"✓ Features standardized (mean=0, std=1)")

# ============================================
# PHASE 5: TRAIN-TEST SPLIT
# ============================================
print("\n" + "="*80)
print("PHASE 5: TRAIN-TEST SPLIT")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✓ Train set: {len(X_train)} samples")
print(f"✓ Test set: {len(X_test)} samples")
print(f"✓ Train readmission rate: {y_train.mean():.1%}")
print(f"✓ Test readmission rate: {y_test.mean():.1%}")

# ============================================
# PHASE 6: MODEL TRAINING
# ============================================
print("\n" + "="*80)
print("PHASE 6: MODEL TRAINING & EVALUATION")
print("="*80)

results = {}

# Model 1: Logistic Regression
print("\n✓ Training Logistic Regression...")

lr = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    solver='lbfgs'
)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

lr_auc = roc_auc_score(y_test, lr_prob)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_acc = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)

results['Logistic Regression'] = {
    'AUC': lr_auc,
    'Recall': lr_recall,
    'Precision': lr_precision,
    'F1': lr_f1,
    'Accuracy': lr_acc,
    'Model': lr,
    'Predictions': lr_pred,
    'Probabilities': lr_prob
}

print(f"  AUC: {lr_auc:.4f}")
print(f"  Recall: {lr_recall:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  F1-Score: {lr_f1:.4f}")
print(f"  Accuracy: {lr_acc:.4f}")

# Cross-validation
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='roc_auc')
print(f"  CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Model 2: Random Forest (Tuned)
print("\n✓ Training Random Forest (GridSearchCV)...")

param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [5, 8, 10, 12],
    'min_samples_leaf': [3, 5, 8],
    'min_samples_split': [5, 10, 15],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

rf_grid.fit(X_train, y_train)

print(f"  Best parameters: {rf_grid.best_params_}")
print(f"  Best CV AUC: {rf_grid.best_score_:.4f}")

rf = rf_grid.best_estimator_
rf_pred = rf.predict(X_test)

y_prob = rf_model.predict_proba(X_test)[:, 1]
threshold = 0.35
rf_pred = (y_prob >= threshold).astype(int)

rf_prob = rf.predict_proba(X_test)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_acc = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)

results['Random Forest'] = {
    'AUC': rf_auc,
    'Recall': rf_recall,
    'Precision': rf_precision,
    'F1': rf_f1,
    'Accuracy': rf_acc,
    'Model': rf,
    'Predictions': rf_pred,
    'Probabilities': rf_prob
}

print(f"  AUC: {rf_auc:.4f}")
print(f"  Recall: {rf_recall:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  F1-Score: {rf_f1:.4f}")
print(f"  Accuracy: {rf_acc:.4f}")

# Model 3: Gradient Boosting
print("\n✓ Training Gradient Boosting...")

gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_prob = gb.predict_proba(X_test)[:,1]

gb_auc = roc_auc_score(y_test, gb_prob)
gb_recall = recall_score(y_test, gb_pred)
gb_f1 = f1_score(y_test, gb_pred)
gb_acc = accuracy_score(y_test, gb_pred)
gb_precision = precision_score(y_test, gb_pred)

results['Gradient Boosting'] = {
    'AUC': gb_auc,
    'Recall': gb_recall,
    'Precision': gb_precision,
    'F1': gb_f1,
    'Accuracy': gb_acc,
    'Model': gb,
    'Predictions': gb_pred,
    'Probabilities': gb_prob
}

print(f"  AUC: {gb_auc:.4f}")
print(f"  Recall: {gb_recall:.4f}")
print(f"  Precision: {gb_precision:.4f}")
print(f"  F1-Score: {gb_f1:.4f}")
print(f"  Accuracy: {gb_acc:.4f}")

# ============================================
# PHASE 7: MODEL COMPARISON & SELECTION
# ============================================
print("\n" + "="*80)
print("PHASE 7: MODEL COMPARISON & SELECTION")
print("="*80)

# Create comparison table
comparison_data = {
    'Model': [],
    'AUC': [],
    'Recall': [],
    'Precision': [],
    'F1-Score': [],
    'Accuracy': []
}

for model_name, metrics in results.items():
    comparison_data['Model'].append(model_name)
    comparison_data['AUC'].append(f"{metrics['AUC']:.4f}")
    comparison_data['Recall'].append(f"{metrics['Recall']:.4f}")
    comparison_data['Precision'].append(f"{metrics['Precision']:.4f}")
    comparison_data['F1-Score'].append(f"{metrics['F1']:.4f}")
    comparison_data['Accuracy'].append(f"{metrics['Accuracy']:.4f}")

df_comparison = pd.DataFrame(comparison_data)

print("\n" + df_comparison.to_string(index=False))

# Select best model
best_model_name = max(results.items(), key=lambda x: x[1]['AUC'])[0]
best_auc = results[best_model_name]['AUC']
best_recall = results[best_model_name]['Recall']

print(f"\n{'='*80}")
print(f"FINAL RECOMMENDATION")
print(f"{'='*80}")

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   AUC: {best_auc:.4f}")
print(f"   Recall: {best_recall:.4f}")

if best_auc >= 0.85:
    print(f"\n✓✓✓ TARGET ACHIEVED: AUC {best_auc:.4f} >= 0.85 ✓✓✓")
elif best_auc >= 0.80:
    print(f"\n⚠️ CLOSE TO TARGET: AUC {best_auc:.4f} (Need 0.85)")
else:
    print(f"\n❌ BELOW TARGET: AUC {best_auc:.4f}")
    print(f"\nROOT CAUSES:")
    print(f"1. Features may not be predictive enough")
    print(f"2. Data quality issue (errors in target or features)")
    print(f"3. Class imbalance too severe ({y.mean():.1%} positive)")
    print(f"4. Need more training data")

print(f"\n{'='*80}")

PROFESSIONAL ML SOLUTION: AUC > 85% FOR MATERNITY READMISSION

PHASE 0: DATA EXPLORATION & VALIDATION (CRITICAL)

✓ Data loaded: 1000 rows, 8 columns

✓ Column names in your CSV:
['Age', 'DeliveryType', 'Complications', 'Comorbidities', 'LOS', 'DaysToFollowup', 'Location', 'Readmitted']

✓ Data types:
Age                 int64
DeliveryType        int64
Complications       int64
Comorbidities       int64
LOS               float64
DaysToFollowup      int64
Location            int64
Readmitted          int64
dtype: object

✓ Target variable (checking for 'Readmitted' or similar):
  Found: Readmitted
  Values: [1 0]
  Distribution: Readmitted
1    500
0    500
Name: count, dtype: int64

✓ Missing values:
  No missing values

✓ Numeric columns:
['Age', 'DeliveryType', 'Complications', 'Comorbidities', 'LOS', 'DaysToFollowup', 'Location', 'Readmitted']

✓ Categorical columns:
[]

PHASE 1: DATA PREPARATION (ENCODING + CLEANING)

✓ Step 1: Encoding categorical variables

✓ Step 2: Encoding tar

NameError: name 'rf_model' is not defined